# Clase 5 · 02 · Tus experimentos

Ahora experimentás vos. **No hace falta escribir código:** arriba del notebook hay widgets para elegir el modelo, las variables y los hiperparámetros. Cada vez que ejecutás la celda **"Entrenar y registrar"** se crea un run nuevo en tu experimento.

**Consigna mínima:** al menos **5 runs** distintos en la sección 1 y **un barrido** en la sección 3. Las preguntas de la entrega te piden comparar cosas concretas; abajo hay una lista de experimentos sugeridos que las cubren.

| Widget | Opciones | Qué controla |
|---|---|---|
| `03 - Modelo` | `logistic_regression`, `random_forest`, `gradient_boosting` | El algoritmo |
| `04 - Features` | `solo_regla`, `basicas`, `cliente`, `completas` | Qué columnas ve el modelo (ver tabla abajo) |
| `05 - class_weight` | `ninguno`, `balanced` | Si los contracargos pesan más al entrenar |
| `06 - Umbral` | `0.5` … `0.2` | Desde qué probabilidad se genera una alerta |
| `07 - C` | `0.01` … `10` | Sólo logística: menos regularización cuanto más grande |
| `08 - n_estimators` | `50`, `100`, `200` | Sólo bosque: cantidad de árboles |
| `09 - max_depth` | `3`, `5`, `8`, `sin_limite` | Bosque y boosting: profundidad máxima de cada árbol |
| `10 - learning_rate` | `0.03`, `0.1`, `0.3` | Sólo boosting: tamaño de cada paso |
| `11 - Nombre del run` | texto libre | Opcional; si lo dejás vacío se genera uno |
| `12 - Barrido` | un hiperparámetro | Qué variar en la sección 3 |

| Conjunto de features | Columnas |
|---|---|
| `solo_regla` | `rule_alert` (la alerta de las clases anteriores) |
| `basicas` | `amount`, `payment_channel` |
| `cliente` | las básicas + `hour`, `account_age_days`, `segment`, `country` |
| `completas` | las de cliente + `is_night`, `day_of_week`, `category`, `price`, `amount_to_price`, `device_customers` |

> Si al cambiar un widget la celda se ejecuta sola, no pasa nada: cada ejecución es un run más. Para evitarlo, en el ícono de engranaje de los widgets elegí **On widget change → Do nothing**.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/ml_lab

In [ ]:
dbutils.widgets.dropdown("modelo", "logistic_regression", WIDGET_CHOICES["modelo"], "03 - Modelo")
dbutils.widgets.dropdown("features", "basicas", WIDGET_CHOICES["features"], "04 - Features")
dbutils.widgets.dropdown("class_weight", "ninguno", WIDGET_CHOICES["class_weight"], "05 - class_weight")
dbutils.widgets.dropdown("umbral", "0.5", WIDGET_CHOICES["umbral"], "06 - Umbral")
dbutils.widgets.dropdown("C", "1", WIDGET_CHOICES["C"], "07 - C")
dbutils.widgets.dropdown("n_estimators", "100", WIDGET_CHOICES["n_estimators"], "08 - n_estimators")
dbutils.widgets.dropdown("max_depth", "8", WIDGET_CHOICES["max_depth"], "09 - max_depth")
dbutils.widgets.dropdown("learning_rate", "0.1", WIDGET_CHOICES["learning_rate"], "10 - learning_rate")
dbutils.widgets.text("nombre_run", "", "11 - Nombre del run")
dbutils.widgets.dropdown("barrido", "max_depth", ["max_depth", "n_estimators", "C", "learning_rate", "umbral", "features", "modelo"], "12 - Barrido")

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
spark.conf.set("spark.sql.session.timeZone", "UTC")
T = lambda name: qualified_table(config, name)
client = MlflowClient()

In [ ]:
assert spark.catalog.tableExists(T("ml_features")), "Falta ml_features. Ejecutá primero 00_preflight."
experiment = setup_experiment(spark, config)
data_version = table_version(spark, T("ml_features"))
print("Experimento:", experiment.name)

train = load_split(spark, config, "train")
test = load_split(spark, config, "test")
print(f"train: {len(train):,} filas · test: {len(test):,} filas · contracargos en test: {test['chargeback'].mean():.1%}")

def current_choice():
    """Lee los widgets y devuelve la configuración del run."""
    choice = {
        "model_type": dbutils.widgets.get("modelo"),
        "feature_set": dbutils.widgets.get("features"),
        "class_weight": parse_param("class_weight", dbutils.widgets.get("class_weight")),
        "threshold": parse_param("umbral", dbutils.widgets.get("umbral")),
    }
    for name in MODEL_PARAMS[choice["model_type"]]:
        choice[name] = parse_param(name, dbutils.widgets.get(name))
    return choice

## 1. Entrenar y registrar

Elegí los valores en los widgets y ejecutá **sólo la celda siguiente**. Repetí cambiando los widgets. Los hiperparámetros que no corresponden al modelo elegido se ignoran (por ejemplo, `C` en un bosque).

In [ ]:
choice = current_choice()
name = dbutils.widgets.get("nombre_run").strip() or f"{choice['model_type']}-{choice['feature_set']}-u{choice['threshold']}"
result = train_and_log(train, test, config, "02", run_name=name, data_version=data_version, **choice)

print(f"Run '{name}' registrado · run_id {result['run_id']}")
print("Configuración:", {k: v for k, v in choice.items()})
display(pd.DataFrame([result["metrics"]]).round(3))
for fig in result["figures"].values():
    display(fig)

**Experimentos sugeridos** (cada uno responde una pregunta de la entrega):

1. **Línea de base:** `logistic_regression` + `solo_regla`. ¿Un modelo mejora a la regla de siempre?
2. **Más información:** el mismo modelo con `basicas`, `cliente` y `completas`. ¿Cuánto aportan las variables?
3. **Otro algoritmo:** `random_forest` y `gradient_boosting` con `completas`.
4. **Desbalance:** repetí tu mejor configuración con `class_weight = balanced`. Mirá precision y recall.
5. **Umbral:** repetí tu mejor configuración con umbral `0.3` o `0.2`. ¿Cambia el PR-AUC? ¿Y la cantidad de alertas?

## 2. Tus runs hasta ahora

Ordenados del mejor al peor PR-AUC. También podés verlos en la interfaz del experimento.

In [ ]:
runs = my_runs(config, experiment.experiment_id)
print(f"Tenés {len(runs)} runs de modelos registrados.")
display(runs.drop(columns=["model_uri", "run_id"]))

## 3. Barrido de un hiperparámetro

Un barrido entrena varias veces **la misma configuración de los widgets** cambiando un solo valor: el que elegiste en `12 - Barrido`. Todas las ejecuciones quedan agrupadas como *runs hijos* (*nested runs*) dentro de un run padre, así no se mezclan con el resto en la interfaz.

Sugerencia: `random_forest` con `completas` y barrido sobre `max_depth`. Mirá qué pasa con `train_pr_auc` y con el PR-AUC de test cuando el árbol puede crecer sin límite.

In [ ]:
base = current_choice()
param = dbutils.widgets.get("barrido")
key = {"umbral": "threshold", "features": "feature_set", "modelo": "model_type"}.get(param, param)
if param not in ("umbral", "features", "modelo") and param not in MODEL_PARAMS[base["model_type"]]:
    raise ValueError(f"{param} no es un hiperparámetro de {base['model_type']}. Elegí otro barrido o cambiá el modelo.")
values = WIDGET_CHOICES[param]

sweep = []
with mlflow.start_run(run_name=f"barrido_{param}", tags={**base_tags(config, "02"), "tipo": "barrido", "barrido": param}):
    mlflow.log_params({k: ("sin_limite" if v is None else v) for k, v in base.items() if k != key})
    for value in values:
        trial = dict(base)
        if param == "modelo":
            trial = {k: v for k, v in trial.items() if k in ("feature_set", "class_weight", "threshold")}
            trial.update({name: parse_param(name, dbutils.widgets.get(name)) for name in MODEL_PARAMS[value]})
        trial[key] = parse_param(param, value)
        result = train_and_log(train, test, config, "02", run_name=f"{param}={value}", nested=True,
                               extra_tags={"barrido": param}, data_version=data_version, **trial)
        sweep.append({param: value, **result["metrics"]})
        print(f"{param}={value}: PR-AUC test {result['metrics']['test_pr_auc']:.3f} · train {result['metrics']['train_pr_auc']:.3f}")

sweep = pd.DataFrame(sweep)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(sweep[param], sweep["train_pr_auc"], marker="o", color="#f59e0b", label="train")
axes[0].plot(sweep[param], sweep["test_pr_auc"], marker="o", color="#2563eb", label="test")
axes[0].set_title("PR-AUC")
axes[1].plot(sweep[param], sweep["test_precision"], marker="o", color="#16a34a", label="precision")
axes[1].plot(sweep[param], sweep["test_recall"], marker="o", color="#dc2626", label="recall")
axes[1].set_title("Precision y recall en test")
for ax in axes:
    ax.set_xlabel(param)
    ax.legend()
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()
display(sweep.round(3))

**Para observar:** si la métrica de **train** sigue subiendo mientras la de **test** baja, el modelo está **sobreajustando**: memoriza los datos de entrenamiento en lugar de aprender patrones que se repitan. Registrar las dos métricas en cada run es lo que permite verlo.

En la interfaz del experimento, seleccioná los runs hijos del barrido y usá **Compare → Parallel coordinates** o **Scatter plot** para ver lo mismo con las herramientas de MLflow.

Cuando tengas tus runs, seguí con `03_comparar_y_elegir`.